# Data preprocessing — retain and estimate family size

This independent variant retains `famsize` and applies the provisional scenario
evaluated in `data_exploration_famsize.ipynb`. All other existing treatments are
carried over. Run from the project root or `notebooks/`; requires pandas and NumPy.
Outputs go to `outputs/famsize_scenario/`, separately from the existing pipeline.
Assistant execution does not verify the selected VS Code kernel.


### Environment check

The Python interpreter, version, working directory, and pandas version
are displayed to record the execution environment.

In [1]:
import sys
import os
import pandas as pd
from pathlib import Path

print(sys.executable)
print(sys.version)
print(os.getcwd())
print(pd.__version__)

C:\Users\alver\AppData\Local\Programs\Python\Python314\python.exe
3.14.0 (tags/v3.14.0:ebf955d, Oct  7 2025, 10:15:03) [MSC v.1944 64 bit (AMD64)]
C:\Data-Mining-Project\notebooks
2.3.3


### Data loading and initial dimensions

The CSV is loaded into the DataFrame `data`.
The loading options preserve the literal education category "None"
while treating empty fields as missing values.

The imported dataset contains 649 rows and 34 columns.
One column is an exported row index, so these are not 34 analytical features.

In [2]:
from pathlib import Path
project_root = Path.cwd().resolve()
if not (project_root / 'data/raw/student_data.csv').exists():
    project_root = project_root.parent
assert (project_root / 'data/raw/student_data.csv').exists(), 'Open the project root or notebooks directory.'
data = pd.read_csv(
    project_root / 'data/raw/student_data.csv',
    keep_default_na=False,
    na_values=['']
)
student_data = data.drop(columns=['Unnamed: 0'])
assert len(student_data) == len(data)
data.shape

Out[0]: (649, 34)


(649, 34)

### Student variables

The exported row index is excluded to create the table of student
variables used for preprocessing.

In [3]:
student_data = data.drop(columns=['Unnamed: 0'])

student_data.shape

Out[0]: (649, 33)


(649, 33)

### Missing values before preprocessing

Missing counts and percentages are recorded before applying the selected treatment.

In [4]:
missing_counts = data.isna().sum()
missing_percentages = (missing_counts / len(data) * 100).round(2)

missing_summary = pd.DataFrame({
    'Missing count': missing_counts,
    'Missing (%)': missing_percentages
})

missing_summary

Out[0]: 
            Missing count  Missing (%)
Unnamed: 0              0         0.00
school                  1         0.15
sex                    10         1.54
age                     0         0.00
address                17         2.62
famsize                 3         0.46
Pstatus                28         4.31
Medu                   27         4.16
Fedu                   19         2.93
Mjob                    6         0.92
Fjob                   18         2.77
reason                  7         1.08
guardian                8         1.23
traveltime             11         1.69
studytime              14         2.16
failures              551        84.90
schoolsup              30         4.62
famsup                  7         1.08
paid                   31         4.78
activities             31         4.78
nursery                20         3.08
higher                 21         3.24
internet                5         0.77
romantic                1         0.15
famrel          

,Missing count,Missing (%)
Unnamed: 0,0,0.00
school,1,0.15
sex,10,1.54
age,0,0.00
address,17,2.62
famsize,3,0.46
Pstatus,28,4.31
Medu,27,4.16
Fedu,19,2.93
Mjob,6,0.92


### Treatment of missing absences and failures

A working copy of the dataset is created for preprocessing.
Missing absences and previous failures are replaced with zero,
assuming that blanks represent no absences or failures.
This interpretation remains subject to source confirmation.

In [5]:
student_data_clean = student_data.copy()

student_data_clean['absences'] = student_data_clean['absences'].fillna(0)
student_data_clean['failures'] = student_data_clean['failures'].fillna(0)

### Validation of zero replacement

Before-and-after counts verify the replacement of missing absences and failures with zero.

In [6]:
replacement_check = pd.DataFrame({
    'Missing before': student_data[['absences', 'failures']].isna().sum(),
    'Missing after': student_data_clean[['absences', 'failures']].isna().sum(),
    'Zeros after': student_data_clean[['absences', 'failures']].eq(0).sum()
})

replacement_check

Out[0]: 
          Missing before  Missing after  Zeros after
absences             260              0          260
failures             551              0          551


,Missing before,Missing after,Zeros after
absences,260,0,260
failures,551,0,551


### Row preservation and remaining missing values

Row counts and order are checked after filling. Remaining missing values
are displayed for later preprocessing decisions.

In [7]:
assert student_data_clean.shape == student_data.shape
assert student_data_clean.index.equals(student_data.index)
assert student_data_clean[['absences', 'failures']].isna().sum().sum() == 0

student_data_clean.isna().sum()

Out[0]: 
school         1
sex           10
age            0
address       17
famsize        3
Pstatus       28
Medu          27
Fedu          19
Mjob           6
Fjob          18
reason         7
guardian       8
traveltime    11
studytime     14
failures       0
schoolsup     30
famsup         7
paid          31
activities    31
nursery       20
higher        21
internet       5
romantic       1
famrel        24
freetime      20
goout         31
Dalc          24
Walc          27
health        17
absences       0
G1             0
G2             0
G3             0
dtype: int64


school         1
sex           10
age            0
address       17
famsize        3
Pstatus       28
Medu          27
Fedu          19
Mjob           6
Fjob          18
reason         7
guardian       8
traveltime    11
studytime     14
failures       0
schoolsup     30
famsup         7
paid          31
activities    31
nursery       20
higher        21
internet       5
romantic       1
famrel        24
freetime      20
goout         31
Dalc          24
Walc          27
health        17
absences       0
G1             0
G2             0
G3             0
dtype: int64

### Order of remaining treatments

Remaining missing values are sorted from most to least frequent.
Variables are treated in this order, and variables of the same type
share one treatment.

In [8]:
remaining_missing = student_data_clean.isna().sum()

remaining_missing = remaining_missing[remaining_missing > 0].sort_values(
    ascending=False
)

remaining_missing

Out[0]: 
goout         31
paid          31
activities    31
schoolsup     30
Pstatus       28
Medu          27
Walc          27
famrel        24
Dalc          24
higher        21
freetime      20
nursery       20
Fedu          19
Fjob          18
health        17
address       17
studytime     14
traveltime    11
sex           10
guardian       8
famsup         7
reason         7
Mjob           6
internet       5
famsize        3
school         1
romantic       1
dtype: int64


goout         31
paid          31
activities    31
schoolsup     30
Pstatus       28
Medu          27
Walc          27
famrel        24
Dalc          24
higher        21
freetime      20
nursery       20
Fedu          19
Fjob          18
health        17
address       17
studytime     14
traveltime    11
sex           10
guardian       8
famsup         7
reason         7
Mjob           6
internet       5
famsize        3
school         1
romantic       1
dtype: int64

### Treatment of missing Yes/No responses

The eight Yes/No features contain between 1 (`romantic`) and 31
(`paid`, `activities`) missing responses. Missing responses are recorded
as a separate "Unknown" category, so no answer is guessed for a student.

`data_exploration.ipynb` (Missing-value overlap in Yes/No features) shows
that missing responses rarely occur together. Of the 140 students missing
at least one Yes/No response, 135 are missing exactly one, and the largest
pairwise overlap involves two students. "Unknown" therefore marks
individual unanswered questions rather than a group of students who left
the questionnaire incomplete.

"Unknown" adds a third level to each Yes/No feature. This must be
considered when the features are encoded.

In [9]:
yes_no_columns = [
    'schoolsup', 'famsup', 'paid', 'activities',
    'nursery', 'higher', 'internet', 'romantic'
]

for column in yes_no_columns:
    student_data_clean[column] = student_data_clean[column].fillna('Unknown')

### Validation of Yes/No replacement

Before-and-after counts verify that every missing response became
"Unknown" and that recorded "yes" and "no" answers are unchanged.

In [10]:
yes_no_check = pd.DataFrame({
    'Missing before': student_data[yes_no_columns].isna().sum(),
    'Missing after': student_data_clean[yes_no_columns].isna().sum(),
    'Unknown after': student_data_clean[yes_no_columns].eq('Unknown').sum()
})

assert student_data_clean[yes_no_columns].eq('yes').sum().equals(
    student_data[yes_no_columns].eq('yes').sum()
)
assert student_data_clean[yes_no_columns].eq('no').sum().equals(
    student_data[yes_no_columns].eq('no').sum()
)

yes_no_check

Out[0]: 
            Missing before  Missing after  Unknown after
schoolsup               30              0             30
famsup                   7              0              7
paid                    31              0             31
activities              31              0             31
nursery                 20              0             20
higher                  21              0             21
internet                 5              0              5
romantic                 1              0              1


,Missing before,Missing after,Unknown after
schoolsup,30,0,30
famsup,7,0,7
paid,31,0,31
activities,31,0,31
nursery,20,0,20
higher,21,0,21
internet,5,0,5
romantic,1,0,1


### Study time by school

Missing study times are counted by school, and the median study time of
each school is calculated from recorded values. Every student with a
missing study time has a recorded school.

In [11]:
study_by_school = pd.DataFrame({
    'Missing studytime': student_data_clean.loc[
        student_data_clean['studytime'].isna(), 'school'
    ].value_counts(),
    'Median studytime': student_data_clean.groupby('school')['studytime'].median()
})

study_by_school

Out[0]: 
        Missing studytime  Median studytime
school                                     
GP                      8             203.5
MS                      6             156.0


,Missing studytime,Median studytime
school,,
GP,8,203.5
MS,6,156.0


### Treatment of missing study time

GP students report more weekly study time than MS students, so a single
overall median would understate study time at GP and overstate it at MS.
Each missing value is replaced with the median of the student's school.
The median is used because study time is strongly right-skewed.

`transform('median')` returns, for every student, the median of their
school, aligned with the original rows. `fillna` uses that value only
where study time is missing.

In [12]:
school_study_median = student_data_clean.groupby('school')['studytime'].transform('median')

student_data_clean['studytime'] = student_data_clean['studytime'].fillna(school_study_median)

### Validation of study-time replacement

The filled values are displayed by school. The checks confirm that no
study time is missing and that recorded values are unchanged.

In [13]:
filled_study = student_data_clean.loc[
    student_data['studytime'].isna(),
    ['school', 'studytime']
]

assert student_data_clean['studytime'].isna().sum() == 0
assert student_data_clean['studytime'].where(
    student_data['studytime'].notna()
).equals(student_data['studytime'])

filled_study.value_counts()

Out[0]: 
school  studytime
GP      203.5        8
MS      156.0        6
Name: count, dtype: int64


school  studytime
GP      203.5        8
MS      156.0        6
Name: count, dtype: int64

### Travel time by school and residence

Missing travel times are counted by school and residence type, and the
median travel time of each combination is calculated from recorded values.
Every student with a missing travel time has a recorded school and
residence. `data_exploration.ipynb` (Travel time, residence, and school)
shows that rural students travel longer than urban students within both
schools.

In [14]:
travel_by_group = pd.DataFrame({
    'Missing traveltime': student_data_clean.loc[
        student_data_clean['traveltime'].isna(), ['school', 'address']
    ].value_counts(),
    'Median traveltime': student_data_clean.groupby(
        ['school', 'address']
    )['traveltime'].median()
})

travel_by_group['Missing traveltime'] = (
    travel_by_group['Missing traveltime'].fillna(0).astype(int)
)

travel_by_group

Out[0]: 
                Missing traveltime  Median traveltime
school address                                       
GP     R                         1               19.0
       U                         7                9.0
MS     R                         3               25.0
       U                         0               17.0


Missing traveltime  Median traveltime
school address                                       
GP     R                         1               19.0
       U                         7                9.0
MS     R                         3               25.0
       U                         0               17.0

### Treatment of missing travel time

Travel time depends on the school and on where the student lives, so a
single overall median would understate rural journeys and overstate urban
ones. Each missing value is replaced with the median of students from the
same school and residence type.

In [15]:
group_travel_median = student_data_clean.groupby(
    ['school', 'address']
)['traveltime'].transform('median')

student_data_clean['traveltime'] = student_data_clean['traveltime'].fillna(group_travel_median)

### Validation of travel-time replacement

The filled values are displayed by school and residence. The checks
confirm that no travel time is missing and that recorded values are
unchanged.

In [16]:
filled_travel = student_data_clean.loc[
    student_data['traveltime'].isna(),
    ['school', 'address', 'traveltime']
]

assert student_data_clean['traveltime'].isna().sum() == 0
assert student_data_clean['traveltime'].where(
    student_data['traveltime'].notna()
).equals(student_data['traveltime'])

filled_travel.value_counts()

Out[0]: 
school  address  traveltime
GP      U        9.0           7
MS      R        25.0          3
GP      R        19.0          1
Name: count, dtype: int64


school  address  traveltime
GP      U        9.0           7
MS      R        25.0          3
GP      R        19.0          1
Name: count, dtype: int64

### Rating medians

The median of each 1–5 rating feature is calculated from recorded
responses before filling.

In [17]:
rating_columns = [
    'famrel', 'freetime', 'goout',
    'Dalc', 'Walc', 'health'
]

rating_medians = student_data_clean[rating_columns].median()

rating_medians

Out[0]: 
famrel      4.0
freetime    3.0
goout       3.0
Dalc        1.0
Walc        2.0
health      4.0
dtype: float64


famrel      4.0
freetime    3.0
goout       3.0
Dalc        1.0
Walc        2.0
health      4.0
dtype: float64

### Treatment of missing ratings

The six rating features contain between 17 (`health`) and 31 (`goout`)
missing responses. Each missing response is replaced with the median of
its feature. The median of an ordered 1–5 scale is always a valid
response level and is not affected by extreme responses.

`data_exploration.ipynb` (Missing-value overlap in rating features) shows
that missing ratings rarely occur together. Of the 129 students missing
at least one rating, 115 are missing exactly one, and the largest
pairwise overlap involves three students. Each feature can therefore be
filled separately.

In [18]:
for column in rating_columns:
    student_data_clean[column] = student_data_clean[column].fillna(
        rating_medians[column]
    )

### Validation of rating replacement

Before-and-after counts verify that no rating is missing, that every
filled value equals the feature median, and that recorded responses are
unchanged.

In [19]:
rating_check = pd.DataFrame({
    'Missing before': student_data[rating_columns].isna().sum(),
    'Missing after': student_data_clean[rating_columns].isna().sum(),
    'Median used': rating_medians
})

assert student_data_clean[rating_columns].isna().sum().sum() == 0
assert student_data_clean[rating_columns].where(
    student_data[rating_columns].notna()
).equals(student_data[rating_columns])

rating_check

Out[0]: 
          Missing before  Missing after  Median used
famrel                24              0          4.0
freetime              20              0          3.0
goout                 31              0          3.0
Dalc                  24              0          1.0
Walc                  27              0          2.0
health                17              0          4.0


,Missing before,Missing after,Median used
famrel,24,0,4.0
freetime,20,0,3.0
goout,31,0,3.0
Dalc,24,0,1.0
Walc,27,0,2.0
health,17,0,4.0


### Treatment of missing categorical values

Missing parental status, parental education, parental occupation,
reason for choosing the school, and residence type are recorded as a
separate unknown category rather than guessed. The observed values give
no basis for inferring, for example, a parent's occupation or education.

- `Pstatus` uses "U", following the single-letter coding of "T"
  (together) and "A" (apart).
- The other features use "Unknown". In `address`, "U" already means
  urban, so a single letter would merge unknown residences with urban
  ones.
- In `Medu` and `Fedu`, "Unknown" remains distinct from the recorded
  level "None" (no formal education).

The unknown category has no position on the education scale. This must
be considered when parental education is encoded.

In [20]:
student_data_clean['Pstatus'] = student_data_clean['Pstatus'].fillna('U')

unknown_columns = ['Medu', 'Fedu', 'Fjob', 'address', 'reason', 'Mjob']

for column in unknown_columns:
    student_data_clean[column] = student_data_clean[column].fillna('unknown')

### Validation of categorical replacement

Before-and-after counts verify that every missing value became the
unknown category and that recorded categories are unchanged.

In [21]:
category_columns = ['Pstatus'] + unknown_columns

unknown_after = student_data_clean[unknown_columns].eq('unknown').sum()
unknown_after['Pstatus'] = student_data_clean['Pstatus'].eq('U').sum()

category_check = pd.DataFrame({
    'Missing before': student_data[category_columns].isna().sum(),
    'Missing after': student_data_clean[category_columns].isna().sum(),
    'Unknown after': unknown_after
}).loc[category_columns]

assert student_data_clean[category_columns].isna().sum().sum() == 0
assert student_data_clean[category_columns].where(
    student_data[category_columns].notna()
).equals(student_data[category_columns])

category_check

Out[0]: 
         Missing before  Missing after  Unknown after
Pstatus              28              0             28
Medu                 27              0             27
Fedu                 19              0             19
Fjob                 18              0             18
address              17              0             17
reason                7              0              7
Mjob                  6              0              6


,Missing before,Missing after,Unknown after
Pstatus,28,0,28
Medu,27,0,27
Fedu,19,0,19
Fjob,18,0,18
address,17,0,17
reason,7,0,7
Mjob,6,0,6


### Treatment of missing sex

Ten students have no recorded sex. The missing values are recorded as a
separate category, "U", rather than assigned to the more frequent
category. Following the single-letter coding of "F" and "M", "U" marks
an unknown value, as in `Pstatus`.

In [22]:
student_data_clean['sex'] = student_data_clean['sex'].fillna('U')

### Validation of sex replacement

Category counts before and after filling are compared.

In [23]:
sex_check = pd.DataFrame({
    'Before': student_data['sex'].value_counts(dropna=False),
    'After': student_data_clean['sex'].value_counts(dropna=False)
}).fillna(0).astype(int)

assert student_data_clean['sex'].isna().sum() == 0

sex_check

Out[0]: 
     Before  After
sex               
F       377    377
M       262    262
U         0     10
NaN      10      0


,Before,After
sex,,
F,377,377
M,262,262
U,0,10
NaN,10,0


### Treatment of missing guardian

Eight students have no recorded guardian. The missing values are replaced
with the most frequent guardian (the mode). Mothers are the recorded
guardian for most students, and eight values are too few to change the
distribution meaningfully.

In [24]:
guardian_mode = student_data_clean['guardian'].mode()[0]

student_data_clean['guardian'] = student_data_clean['guardian'].fillna(guardian_mode)

### Validation of guardian replacement

Guardian counts before and after filling are compared.

In [25]:
guardian_check = pd.DataFrame({
    'Before': student_data['guardian'].value_counts(dropna=False),
    'After': student_data_clean['guardian'].value_counts(dropna=False)
}).fillna(0).astype(int)

assert student_data_clean['guardian'].isna().sum() == 0

guardian_check

Out[0]: 
          Before  After
guardian               
father       150    150
mother       450    458
other         41     41
NaN            8      0


,Before,After
guardian,,
father,150,150
mother,450,458
other,41,41
NaN,8,0


### Family-size scenario and source boundaries

The user reports that the student is excluded from `famsize`, and that the
professor advised correcting questionable values instead of dropping the feature.
We retain the feature and preserve every recorded positive value, including 1.
Zero is treated as an unknown count **for this provisional scenario**; living
alone is not ruled out by these data. Guardian and parental status do not prove
co-residence. Therefore we do not force `Pstatus = T` to imply a size of 2.

We estimate zero/missing values from positive recorded sizes grouped by recorded
`Pstatus` and `guardian`. Missing grouping keys remain missing until explicitly
labelled for matching; we do not use imputed guardians as observed evidence.
Fallback order is joint-group median, parental-status median, then overall
median. No minimum group-size threshold has been approved; group counts and
spread are displayed for review. Donor values are plausible observations, not
independently verified household counts. Fractional estimates are rounded to
the nearest integer, with halves rounded upward, because this is a people count.
These are transparent candidate rules, not reconstructed household facts.

`Medu` and `Fedu` provide context only: education does not establish residence,
and literal `None` denotes no education rather than an absent parent.
Grouped imputation may reinforce associations with parental status/guardian;
review sensitivity before using the feature in clustering.

Course adaptations: `memory/course-reference/exploration.md` (P1 cells 89, 91:
group summaries) and `memory/course-reference/preprocessing.md` (P2 cells 79, 81:
imputation alternatives; P2 cells 56–58: before/after checks). The lab archives
and `memory/prior-experiments.md` are absent in this checkout, so cited source
cells could not be re-inspected. These notes do not establish that the proposed
grouping or zero treatment was prescribed by the course.


### Check duplicate student records

Exclude the exported row index and count duplicate records. This check does not delete students.


In [26]:
duplicate_count = student_data.duplicated().sum()
print('Duplicate records:', int(duplicate_count))


Duplicate records: 0


### Inspect family-size donors from recorded values

Use the raw-loaded `student_data`, not already filled parental/guardian values, to build comparison groups.


In [27]:
family_donors = student_data.loc[
    student_data['famsize'].ge(1), ['famsize', 'Pstatus', 'guardian']
].copy()
family_donors['Pstatus'] = family_donors['Pstatus'].fillna('Missing')
family_donors['guardian'] = family_donors['guardian'].fillna('Missing')
family_donor_summary = family_donors.groupby(['Pstatus', 'guardian'])['famsize'].agg(
    ['count', 'median', 'min', 'max']
)
assert len(family_donors) + student_data['famsize'].eq(0).sum() + student_data['famsize'].isna().sum() == len(student_data)
family_donor_summary


Out[0]: 
                  count  median  min  max
Pstatus guardian                         
A       father        5     8.0  1.0  9.0
        mother       51     3.0  1.0  8.0
        other         7     3.0  1.0  9.0
Missing father       10     6.5  1.0  9.0
        mother       13     8.0  1.0  9.0
        other         2     7.5  6.0  9.0
T       Missing       6     5.5  1.0  8.0
        father      121     5.0  1.0  9.0
        mother      344     5.5  1.0  9.0
        other        30     6.5  1.0  9.0


count  median  min  max
Pstatus guardian                         
A       father        5     8.0  1.0  9.0
        mother       51     3.0  1.0  8.0
        other         7     3.0  1.0  9.0
Missing father       10     6.5  1.0  9.0
        mother       13     8.0  1.0  9.0
        other         2     7.5  6.0  9.0
T       Missing       6     5.5  1.0  8.0
        father      121     5.0  1.0  9.0
        mother      344     5.5  1.0  9.0
        other        30     6.5  1.0  9.0

### Estimate a candidate family size

Join medians to each student using recorded grouping keys. `.fillna` implements the stated fallback order. Add 0.5 and take the floor to round positive half-values upward. The original table stays unchanged.


In [28]:
import numpy as np

family_candidates = student_data[['famsize', 'Pstatus', 'guardian']].copy()
family_candidates['Pstatus'] = family_candidates['Pstatus'].fillna('Missing')
family_candidates['guardian'] = family_candidates['guardian'].fillna('Missing')
joint_medians = family_donors.groupby(['Pstatus', 'guardian'])['famsize'].median()
status_medians = family_donors.groupby('Pstatus')['famsize'].median()
family_candidates = family_candidates.join(joint_medians.rename('Joint median'), on=['Pstatus', 'guardian'])
family_candidates['Status median'] = family_candidates['Pstatus'].map(status_medians)
family_candidates['Overall median'] = family_donors['famsize'].median()
family_candidates['Estimate'] = family_candidates['Joint median'].fillna(
    family_candidates['Status median']
).fillna(family_candidates['Overall median'])
family_candidates['Method'] = 'Joint-group median'
family_candidates.loc[family_candidates['Joint median'].isna(), 'Method'] = 'Parental-status median'
family_candidates.loc[
    family_candidates['Joint median'].isna() & family_candidates['Status median'].isna(), 'Method'
] = 'Overall median'
family_candidates['Proposed'] = np.floor(family_candidates['Estimate'] + 0.5)
family_target = student_data['famsize'].eq(0) | student_data['famsize'].isna()
family_candidates.loc[~family_target, 'Proposed'] = student_data.loc[~family_target, 'famsize']
family_candidates.loc[~family_target, 'Method'] = 'Recorded value retained'
assert family_candidates['Proposed'].notna().all()
assert family_candidates.loc[~family_target, 'Proposed'].equals(student_data.loc[~family_target, 'famsize'])
family_candidates.loc[family_target].groupby(['Pstatus', 'guardian', 'Proposed', 'Method']).size().rename('Rows').to_frame()


Out[0]: 
                                              Rows
Pstatus guardian Proposed Method                  
A       father   8.0      Joint-group median     1
        mother   3.0      Joint-group median    11
        other    3.0      Joint-group median     2
Missing mother   8.0      Joint-group median     3
T       Missing  6.0      Joint-group median     2
        father   5.0      Joint-group median    13
        mother   6.0      Joint-group median    28


Rows
Pstatus guardian Proposed Method                  
A       father   8.0      Joint-group median     1
        mother   3.0      Joint-group median    11
        other    3.0      Joint-group median     2
Missing mother   8.0      Joint-group median     3
T       Missing  6.0      Joint-group median     2
        father   5.0      Joint-group median    13
        mother   6.0      Joint-group median    28

### Review changes before adoption

Compare recorded and candidate distributions and the number of affected students. Only zero/missing values are targets; education and guardian values are not inferred from family size.


In [29]:
family_audit = pd.DataFrame({
    'Original': student_data['famsize'],
    'Proposed': family_candidates['Proposed'],
    'Method': family_candidates['Method'],
    'Targeted': family_target
}, index=student_data.index)
family_distribution = pd.DataFrame({
    'Recorded': student_data['famsize'].value_counts(dropna=False),
    'Candidate': family_candidates['Proposed'].value_counts(dropna=False)
}).fillna(0).astype(int)
assert family_audit.loc[~family_target, 'Original'].equals(family_audit.loc[~family_target, 'Proposed'])
assert student_data.loc[student_data['famsize'].eq(1), 'famsize'].eq(1).all()
print('Targeted rows:', int(family_target.sum()))
family_distribution.sort_index()


Targeted rows: 60
Out[0]: 
     Recorded  Candidate
0.0        57          0
1.0        48         48
2.0        38         38
3.0        48         61
4.0        79         79
5.0        88        101
6.0        62         92
7.0        67         67
8.0        80         84
9.0        79         79
NaN         3          0


,Recorded,Candidate
0.0,57,0
1.0,48,48
2.0,38,38
3.0,48,61
4.0,79,79
5.0,88,101
6.0,62,92
7.0,67,67
8.0,80,84
9.0,79,79


### Apply the candidate and validate preservation

Assign only the candidate family-size Series. Check positive recorded sizes, all other features, row identities, completeness and integer counts. This is a scenario export, not final methodological approval.


In [30]:
before_family_treatment = student_data_clean.copy()
student_data_clean['famsize'] = family_candidates['Proposed']
pd.testing.assert_frame_equal(
    student_data_clean.drop(columns=['famsize']),
    before_family_treatment.drop(columns=['famsize'])
)
assert student_data_clean.index.equals(student_data.index)
assert student_data_clean.shape == student_data.shape
assert student_data_clean['famsize'].ge(1).all()
assert student_data_clean['famsize'].mod(1).eq(0).all()
assert student_data_clean.loc[~family_target, 'famsize'].equals(student_data.loc[~family_target, 'famsize'])
student_data_clean.shape


Out[0]: (649, 33)


(649, 33)

### Treatment of missing school

One student has no recorded school. The value is replaced with the most
frequent school (the mode). With a single missing value, this has no
meaningful effect on the distribution. `.mode()` returns a Series because
several values can tie, so `[0]` selects the first.

In [31]:
school_mode = student_data_clean['school'].mode()[0]

student_data_clean['school'] = student_data_clean['school'].fillna(school_mode)

### Validation of school replacement

School counts before and after filling are compared.

In [32]:
school_check = pd.DataFrame({
    'Before': student_data['school'].value_counts(dropna=False),
    'After': student_data_clean['school'].value_counts(dropna=False)
}).fillna(0).astype(int)

assert student_data_clean['school'].isna().sum() == 0

school_check

Out[0]: 
        Before  After
school               
GP         422    423
MS         226    226
NaN          1      0


,Before,After
school,,
GP,422,423
MS,226,226
NaN,1,0


### Remaining missing values

After all treatments, no missing values should remain in the working
dataset.

In [33]:
assert student_data_clean.isna().sum().sum() == 0

student_data_clean.shape

Out[0]: (649, 33)


(649, 33)

### Working dataset

The prepared table is available as `student_data_clean` in this notebook.
Missing failures and absences are interpreted as zero by user decision.
The source meaning of those blanks remains unconfirmed. Missing Yes/No
responses are recorded as "Unknown". Missing study times are replaced
with the median of the student's school, and missing travel times with
the median of the student's school and residence type. Missing ratings
are replaced with the median of each feature. Missing parental status and
sex are recorded as "U"; missing parental education, parental occupation,
reason, and residence are recorded as "Unknown". Missing guardians and the
missing school are replaced with the mode. `famsize` is retained with the provisional group-median estimates documented above. Further treatments require an explicit decision.
Duplicate records are evaluated above.

### Save the prepared dataset

The processed-data folder is created if needed, then the prepared dataset
is saved with its source row identifier.

In [34]:
output_folder = project_root / 'outputs/famsize_scenario'
output_folder.mkdir(parents=True, exist_ok=True)
output_path = output_folder / 'student_data_preprocessed_famsize.csv'
student_data_clean.to_csv(output_path, index=True, index_label='student_row_id')
family_audit.to_csv(output_folder / 'famsize_correction_audit.csv', index=True, index_label='student_row_id')
reloaded = pd.read_csv(output_path, index_col='student_row_id', keep_default_na=False, na_values=[''])
assert reloaded.shape == student_data_clean.shape
assert reloaded.index.equals(student_data_clean.index)
pd.testing.assert_frame_equal(reloaded.rename_axis(student_data_clean.index.name), student_data_clean, check_dtype=False)
print(output_path)
print('Prepared rows and features:', student_data_clean.shape)
print('Family-size estimates:', int(family_target.sum()))

C:\Data-Mining-Project\outputs\famsize_scenario\student_data_preprocessed_famsize.csv
Prepared rows and features: (649, 33)
Family-size estimates: 60
